# Model development

Two candidate approaches answer the same question: what is the cumulative
probability of live discharge or in-hospital death during days 1–90 after the
day-3 assessment? Multinomial logistic regression provides a simple additive
reference, while histogram gradient boosting can represent nonlinearities and
interactions. This notebook benchmarks and tunes both models using the training
cohort only.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

from src.config import (
    CANDIDATE_FEATURES, DEFAULT_MODEL_PARAMETERS, MODEL_FAMILIES, SEED,
    TUNING_GRIDS,
)
from src.data import create_fixed_split, load_support2
from src.modelling import cross_validate_candidate, default_indicator_features

TABLES = PROJECT_ROOT / 'results' / 'tables'
DEVELOPMENT = PROJECT_ROOT / 'results' / 'development'
TABLES.mkdir(parents=True, exist_ok=True)
DEVELOPMENT.mkdir(parents=True, exist_ok=True)

## Common development cohort and folds

All families use the same eligible patients, raw candidate pool, outcome
definition, horizons, and five patient-level folds. Patient-day rows for the
two discrete-time models are created only after fold assignment.

In [2]:
df = load_support2()
train_df, _, _ = create_fixed_split(df)
train_model = train_df.loc[
    train_df['days_to_hospital_exit'] > 3
].set_index('patient_id').copy()
train_model['event'] = np.where(train_model['in_hospital_death'].eq(1), 2, 1)
fold_labels = (
    train_model['event'].astype(str) + '_'
    + pd.qcut(train_model['days_to_hospital_exit'], q=5, duplicates='drop').astype(str)
)
patient_folds = list(StratifiedKFold(
    5, shuffle=True, random_state=SEED
).split(train_model, fold_labels))
assert all(
    set(train_model.iloc[fit].index).isdisjoint(train_model.iloc[assess].index)
    for fit, assess in patient_folds
)
display(pd.DataFrame({
    'Patients': [len(train_model)], 'Candidate predictors': [len(CANDIDATE_FEATURES)],
    'Patient folds': [len(patient_folds)],
}))

,Patients,Candidate predictors,Patient folds
0,6140,28,5


## Two discrete-time approaches

Both models estimate three conditional states for each day a patient remains at
risk: continued hospitalisation, live discharge, and in-hospital death. The same
probability recursion converts these daily predictions into coherent cumulative
patient-level probabilities, so model comparison isolates functional form rather
than a difference in the prediction target.


## All-feature benchmarks

Initial benchmarks use all 28 original candidates and conservative default
settings. This separates family comparison from later feature simplification.
The primary score is the arithmetic mean of patient-level Brier scores over
days 1–90 and both outcomes.

In [3]:
benchmark_metrics = []
benchmark_folds = []
for family in MODEL_FAMILIES:
    print('Benchmark:', family)
    metrics, folds = cross_validate_candidate(
        train_model, CANDIDATE_FEATURES, family,
        DEFAULT_MODEL_PARAMETERS[family], patient_folds,
        indicator_features=default_indicator_features(CANDIDATE_FEATURES),
    )
    metrics['model'] = family
    folds['model'] = family
    benchmark_metrics.append(metrics)
    benchmark_folds.append(folds)

benchmark_metrics = pd.concat(benchmark_metrics, ignore_index=True)
benchmark_folds = pd.concat(benchmark_folds, ignore_index=True)
benchmark_summary = benchmark_folds.groupby('model')['mean_daily_brier'].agg(
    Mean='mean', SD='std', Count='count',
).sort_values('Mean')
benchmark_summary['SE'] = benchmark_summary['SD'] / np.sqrt(benchmark_summary['Count'])
display(benchmark_summary.round(5))
benchmark_folds.to_csv(DEVELOPMENT / 'all_feature_benchmark_folds.csv', index=False)
benchmark_metrics.to_csv(DEVELOPMENT / 'all_feature_benchmark_metrics.csv', index=False)

Benchmark: Multinomial logistic regression


Benchmark: Histogram gradient boosting


,Mean,SD,Count,SE
model,,,,
Histogram gradient boosting,0.13910,0.00238,5,0.00106
Multinomial logistic regression,0.14105,0.00145,5,0.00065


With all 28 predictors and default settings, histogram gradient boosting
achieves mean daily Brier 0.13910, compared with 0.14105 for multinomial
logistic regression. The modest initial difference supports tuning both models
rather than drawing a conclusion from default configurations.


## Moderate model-specific tuning

Logistic regression varies L2 regularisation on a logarithmic scale. Gradient
boosting uses a short, explicit set of learning rates, leaf counts, iteration
counts, L2 penalties, and minimum leaf sizes rather than a large Cartesian grid.
Every candidate is assessed in the same patient-level folds using mean daily
Brier over days 1–90.


In [4]:
tuning_rows = []
for family in MODEL_FAMILIES:
    for candidate_number, parameters in enumerate(TUNING_GRIDS[family], start=1):
        print(f'Tuning {family}: {candidate_number}/{len(TUNING_GRIDS[family])}')
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            _, folds = cross_validate_candidate(
                train_model, CANDIDATE_FEATURES, family, parameters, patient_folds,
                indicator_features=default_indicator_features(CANDIDATE_FEATURES),
            )
        tuning_rows.append({
            'model': family,
            'candidate': candidate_number,
            'parameters': json.dumps(parameters, sort_keys=True),
            'mean_daily_brier': folds['mean_daily_brier'].mean(),
            'fold_sd': folds['mean_daily_brier'].std(),
            'fold_se': folds['mean_daily_brier'].std() / np.sqrt(len(folds)),
        })

tuning_results = pd.DataFrame(tuning_rows).sort_values(
    ['model', 'mean_daily_brier', 'candidate']
)
tuning_results['rank'] = tuning_results.groupby('model').cumcount() + 1
display(tuning_results.loc[tuning_results['rank'] <= 3].round(5))
tuning_results.to_csv(DEVELOPMENT / 'all_feature_tuning.csv', index=False)

best_parameters = {
    family: json.loads(
        tuning_results.loc[
            tuning_results['model'].eq(family)
            & tuning_results['rank'].eq(1), 'parameters'
        ].iloc[0]
    )
    for family in MODEL_FAMILIES
}
with (DEVELOPMENT / 'best_all_feature_parameters.json').open('w') as file:
    json.dump(best_parameters, file, indent=2)
display(pd.DataFrame([
    {'Model': family, **parameters}
    for family, parameters in best_parameters.items()
]))

Tuning Multinomial logistic regression: 1/6


Tuning Multinomial logistic regression: 2/6


Tuning Multinomial logistic regression: 3/6


Tuning Multinomial logistic regression: 4/6


Tuning Multinomial logistic regression: 5/6


...

Tuning Histogram gradient boosting: 10/12


Tuning Histogram gradient boosting: 11/12


Tuning Histogram gradient boosting: 12/12


,model,candidate,parameters,mean_daily_brier,fold_sd,fold_se,rank
12,Histogram gradient boosting,7,"{""l2_regularization"": 10.0, ""learning_rate"": 0...",0.13658,0.00213,0.00095,1
16,Histogram gradient boosting,11,"{""l2_regularization"": 1.0, ""learning_rate"": 0....",0.13752,0.00088,0.00039,2
17,Histogram gradient boosting,12,"{""l2_regularization"": 10.0, ""learning_rate"": 0...",0.13806,0.00228,0.00102,3
5,Multinomial logistic regression,6,"{""C"": 100.0}",0.14093,0.00136,0.00061,1
2,Multinomial logistic regression,3,"{""C"": 0.1}",0.14097,0.00164,0.00073,2
4,Multinomial logistic regression,5,"{""C"": 10.0}",0.14097,0.00141,0.00063,3


,Model,C,l2_regularization,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf
0,Multinomial logistic regression,100.0,NaN,NaN,NaN,NaN,NaN
1,Histogram gradient boosting,NaN,10.0,0.05,200.0,15.0,50.0


Tuning improves HGB to 0.13658 and logistic regression to 0.14093. The
logistic regularisation curve is effectively flat near its minimum: C=100
improves the score by less than 0.00004 over the next-ranked configuration, so a
wider range is unlikely to change the scientific comparison materially. These
settings define the fixed-complexity starting point for cross-fitted feature
simplification; validation and test results have not influenced them.
